# Calibrate LeakyBucket model using scipy bounded optimisation

Calibrates the single `leakiness` parameter of the LeakyBucket model against
ERA5 forcing and Caravan observed discharge over the calibration period defined
in `settings.json`.

**Objective:** maximise KGE (Kling-Gupta Efficiency) → minimise `1 - KGE`.

**Optimiser:** `scipy.optimize.minimize_scalar` with `method='bounded'`.
One parameter, no heavy dependencies — no `sceua` needed.

**Output:**
- `{path_output}/{caravan_id}_params_leakybucket.csv` — best leakiness value
- `results.json` — calibration KGE and NSE

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

from rich import print

import ewatercycle
import ewatercycle.forcing

try:
    from scripts.leakybucket_model import LeakyBucketLocal
except ImportError:
    sys.path.insert(0, str(Path().resolve().parent))
    from scripts.leakybucket_model import LeakyBucketLocal

In [ ]:
"""On Spider we want to not change our environment every run, so we have to do the pip installing separately.
This means that that is already done and handled by papermill, but since we still want this workflow to work on a SRC machine we need to do the pip installing"""
if "ewater" in str(Path.home()):
    # we are on spider
    run_pips = False
else:
    run_pips = True

In [ ]:
if run_pips:
    # We need the sceua package. If that is not available on your machine,
    # uncomment the line below to install it

    !pip install sceua
    # We make use of the hydrobm package to calculate metrics
    !pip install hydrobm

In [ ]:
import sceua
from hydrobm.metrics import calculate_metric

In [ ]:
# Parameters — overridden by papermill when running on HPC
settings_path = "settings.json"

In [ ]:
with open(settings_path, 'r') as f:
    settings = json.load(f)

display(settings)

In [ ]:
# Skip if calibration output already exists
params_path = Path(settings['path_output']) / (settings['caravan_id'] + '_params_leakybucket.csv')
need_to_run = not params_path.exists()

if not need_to_run:
    display(f'Calibration already complete: {params_path}')

## Load forcing and observations

In [ ]:
# ERA5 forcing (LumpedMakkink) — used as model input
load_location = Path(settings['path_ERA5']) / "work" / "diagnostic" / "script"
ERA5_forcing_object = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=load_location)
display(ERA5_forcing_object)

In [ ]:
# Load the caravan forcing object
caravan_data_object = ewatercycle.forcing.sources['CaravanForcing'].load(directory=settings['path_caravan'])
display(caravan_data_object)


## Calibration objective and model runner

In [ ]:
def calibrationObjective(model_output, observation, start_calibration, end_calibration, metric_name = 'rmse'):
    '''A function that takes in two dataFrames, interpolates the model output to the
    observations and calculates the average absolute difference between the two. '''

    # Combine the two in one dataFrame and interpolate, to make sure times match
    hydro_data = pd.concat([model_output.reindex(observation.index, method = 'ffill'), observation], axis=1,
                           keys=['model','observation'])

    # Only select the calibration period
    hydro_data = hydro_data[hydro_data.index > pd.to_datetime(pd.Timestamp(start_calibration).date())]
    hydro_data = hydro_data[hydro_data.index < pd.to_datetime(pd.Timestamp(end_calibration).date())]

    obs = hydro_data['observation'].to_numpy()
    sim = hydro_data['model'].to_numpy()

    metric_value  = calculate_metric(obs,sim,metric_name)

    return metric_value

    # match metric:
    #     case 'rmse':
    #         # Calculate mean absolute difference
    #         squareDiff = (sim - obs)**2
    #         rmse = np.sqrt(np.mean(squareDiff))
    #         return rmse
    #     case 'nse':
    #         # Calculate Nash Shutcliff Efficiency
    #         nse = 1 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2)
    #         return nse
    #     case 'kge':
    #         # Calculate Kling Gupta Efficiency
    #         if (np.std(obs) == 0) or (np.std(sim) == 0):
    #             r = 0
    #         else:
    #             r = np.corrcoef(obs, sim)[0, 1]
    #         alpha = np.std(sim) / np.std(obs)
    #         beta = np.mean(sim) / np.mean(obs)
    #         kge = 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)
    #         return kge


In [ ]:
# Create an array with parameter values.

# First set minimum and maximum values on the parameters
p_min_initial = np.array([0.001])
p_max_initial = np.array([0.5])

p_initial = (p_min_initial + p_min_initial)/2

# Set initial state values
#               Si,  Su, Sf, Ss, Sp
s_0 = np.array([0])

In [ ]:
# Print parameter names and values for first ensemble member
param_names = ["Leakiness"]
display(list(zip(param_names, np.round(p_initial, decimals=3))))

In [ ]:
# Create a dataframe for the observations
ds_observation = xr.open_mfdataset([caravan_data_object['Q']]).to_pandas()
ds_observation = ds_observation['Q']

In [ ]:
def runModel(params, forcing_object, observations, initial_state, start_calibration, end_calibration):
    # Create model object, notice the forcing object.
    model = LeakyBucketLocal(forcing=forcing_object)

    # Create config file in model.setup()
    config_file, _ = model.setup(leakiness=params[0], initial_storage=initial_state[0],
                                start_time = start_calibration, end_time = end_calibration)
    # Initialize model
    model.initialize(config_file)
    # Run model, capture calculated discharge and timestamps
    Q_m = []
    time = []
    while model.time < model.end_time:
        model.update()
        Q_m.append(model.get_value("discharge")[0])
        time.append(pd.Timestamp(model.time_as_datetime))
    # Finalize model (shuts down container, frees memory)
    model.finalize()

    # Make a pandas series
    model_output = pd.Series(data=Q_m, name="modelled discharge", index=time)

    return model_output

def runModelReturnObjective(params, forcing_object, observations, initial_state, start_calibration, end_calibration):

    model_output = runModel(params, forcing_object, observations, initial_state, start_calibration, end_calibration)

    kge = calibrationObjective(model_output, observations, start_calibration, end_calibration, metric_name = 'kge')

    # kge is best if high, so we do one minus kge to make sure that sce optimizes correctly
    # since sce always minimizes

    score = 1 - kge

    return score

## Run optimisation

In [ ]:
# Define parameter bounds as a sequence of (min, max) pairs
bounds = [(p_min_initial[n], p_max_initial[n]) for n in range(len(p_min_initial))]

display(bounds)

In [ ]:
if need_to_run:
    # Run optimization
    result = sceua.minimize(runModelReturnObjective, bounds, args=(ERA5_forcing_object, ds_observation, s_0, settings['calibration_start_date'],
                                                                  settings['calibration_end_date']), seed=42, max_workers=1)


In [ ]:
if need_to_run:
    # Access the optimization results
    best_params = result.x
    best_function_value = result.fun
    num_iterations = result.nit
    num_function_evaluations = result.nfev

In [ ]:
if need_to_run:
    display(best_function_value)

In [ ]:
if need_to_run:
    model_output = runModel(best_params, ERA5_forcing_object, ds_observation, s_0,
                                              settings['calibration_start_date'], settings['calibration_end_date'])

In [ ]:
if need_to_run:
    # Calculate NSE for the calibration period using the best parameters
    best_nse = calibrationObjective(model_output, ds_observation,
                                    settings['calibration_start_date'], settings['calibration_end_date'],
                                    metric_name='nse')
    display(f"Calibration NSE: {best_nse}")

In [ ]:
if need_to_run:
    # Create a dataframe for the observations
    ds_observation = xr.open_mfdataset([caravan_data_object['Q']]).to_pandas()

## Calibration plot

In [ ]:
if need_to_run:
    # Zoom to 2 years starting from 2000-08-01
    start_time = pd.Timestamp('2000-08-01')
    zoom_end = start_time + pd.DateOffset(years=2)

    obs_zoom = ds_observation["Q"].loc[start_time:zoom_end]
    model_zoom = model_output.loc[start_time:zoom_end]

    # Make a plot of the model output
    obs_zoom.plot(label='observed discharge')
    model_zoom.plot(lw=2.5, label='modelled discharge')
    plt.title(f"Calibrated model (2000-2002)\nKGE: {(1 - best_function_value):.3f}, NSE: {best_nse:.3f}")
    plt.xlabel('Time')
    plt.ylabel("Discharge (mm/d)")
    plt.legend()
    plt.savefig(settings["figure_output"] + "/calibration.png")
    plt.show()

In [ ]:
if need_to_run:
    df_best = model_output

In [ ]:
if need_to_run:
    df_best.index = df_best.index.tz_localize("UTC")
    df_select = df_best.tz_convert("UTC")[settings['validation_start_date']:settings['validation_end_date']]

In [ ]:
if need_to_run:
    # Make a plot of the model output of the minimum value
    ds_observation["Q"].plot(label = 'observed discharge')
    ax = df_select.plot(lw=2.5)
    plt.legend()
    plt.xlim(settings['validation_start_date'],settings['validation_end_date'])

## Save results

In [ ]:
if need_to_run:
    # Save to csv file
    np.savetxt(Path(settings["path_output"]) / (settings['caravan_id'] + "_LB_params_SCE.csv"), best_params, delimiter=",")

In [ ]:
if need_to_run:
    # Save calibration KGE and NSE to results.json (same directory as settings.json)
    results_path = Path(settings["results_path"])
    results = json.loads(results_path.read_text()) if results_path.exists() else {}

    # results["caravan_id"] = settings["caravan_id"]
    results["calibration"] = {
        "KGE": round(1 - best_function_value, 4),
        "NSE": round(best_nse, 4),
        "period": f"{settings['calibration_start_date'][:10]} / {settings['calibration_end_date'][:10]}",
    }

    results_path.write_text(json.dumps(results, indent=4))
    print(f"Calibration KGE = {results['calibration']['KGE']}, NSE = {results['calibration']['NSE']} written to {results_path}")

In [ ]:
# Remove all temporary directories made by the optimization algo.

!rm -rf leakybucketlocal*

In [ ]:
%%capture
if run_pips:
    # Re-install correct numpy for ESMVAlTool
    !pip install esmvaltool